# 9. 텍스트 벡터화, 임베딩과 분류 실습

- 목표: BoW, TF-IDF, N-gram, Word2Vec, 텍스트 분류 베이스라인까지 텍스트를 숫자로 표현하고 활용하는 흐름을 연결합니다.
- 흐름: 8차시에서 만든 텍스트 데이터를 벡터로 바꾸고, 10차시에서는 LLM과 OpenAI 임베딩을 활용한 텍스트마이닝으로 확장합니다.


## 1. 임베딩 (Embedding)

임베딩은 텍스트(단어, 문장, 문서 등)를 **의미를 반영하는 숫자 벡터로 표현**하는 방법입니다.  
단순히 “숫자로 바꿨다”가 아니라, 벡터 공간 위에서 **단어들 사이의 유사도와 관계**가 드러나도록 만드는 것이 핵심입니다.

이전 섹션에서 우리는 텍스트를 숫자로 바꾸기 위한 여러 **인코딩(encoding)** 방법들을 봤습니다.  
이제부터는 한 단계 더 나아가, **의미를 잘 담은 벡터 표현 = 임베딩(embedding)** 을 살펴봅니다.

<img src="image/text_vectorization_sparse_dense.svg" width="760">

이미지 출처: 김민수 강사


### 1.1 왜 임베딩이 필요한가?

#### 1) 기존 인코딩의 한계 (리마인드)

- **정수 인코딩(Integer Encoding)**  
  - 단어마다 번호(ID)만 붙이는 방식입니다.  
    예) `{"강아지": 1, "고양이": 2, "자동차": 3, ...}`  
  - 숫자의 크기와 단어의 의미는 아무 관계가 없는데,  
    모델 입장에서는 `1 < 2 < 3` 같은 **가짜 순서 정보**를 학습할 위험이 있습니다.

- **원-핫 인코딩(One-hot Encoding)**  
  - 단어 사전 크기가 `V`일 때, 길이가 `V`인 벡터에서 해당 단어 위치만 1이고 나머지는 모두 0인 벡터를 사용합니다.
  - 모든 벡터가 서로 직교(orthogonal)라서 **단어 사이의 유사성이 전혀 표현되지 않습니다.**
  - 벡터가 매우 길고 대부분이 0인 **희소(sparse) 벡터**라서, 계산 효율도 좋지 않습니다.

```
어휘 사전: ["강아지", "고양이", "자동차", "사과", "바나나"]
"고양이" = [0, 1, 0, 0, 0]
"강아지" = [1, 0, 0, 0, 0]
→ 두 벡터의 내적 = 0 (완전히 다른 방향)
→ 실제로는 비슷한 동물인데, 벡터 상에서는 아무 관련이 없음
```
정리하면, 기존 인코딩은 “컴퓨터가 처리할 수 있도록 텍스트를 숫자로 바꾸는 것”에는 성공했지만,  
“단어의 의미나 단어들 사이의 관계를 표현하는 것”에는 한계가 있음


#### 2) 임베딩 벡터의 아이디어
- 임베딩은 이런 한계를 극복하기 위해 등장했습니다.
- **벡터의 차원을 줄이고(dense)** 벡터의 각 차원이 단어의 **의미**, 문맥, 역할 등을 반영하도록 학습시키는 것이 목표입니다.
- 즉, 의미가 비슷한 단어는 벡터 공간에서도 가깝게, 의미가 다른 단어는 멀리 떨어지도록 만드는 것입니다.

```
"강아지" = [ 0.13, -0.25,  0.77,  0.05]
"고양이" = [ 0.12, -0.33,  0.80,  0.04]
"자동차" = [-0.80,  0.10, -0.12,  0.55]
→ "강아지"와 "고양이" 벡터는 서로 가깝고, "자동차" 벡터는 이 둘과 멀리 떨어져 있도록 학습

이렇게 되면, 비슷한 단어끼리는 군집(clustering)이 생기고
단어 간 관계를 벡터 연산으로도 어느 정도 표현할 수 있습니다.
(예: “왕 - 남자 + 여자 ≈ 여왕” 같은 Word2Vec의 예시)

```

#### 3) 인코딩 vs 임베딩 비교

| 구분             | 정수 인코딩        | 원-핫 인코딩          | 임베딩(Embedding)              |
|------------------|--------------------|------------------------|--------------------------------|
| 표현 방식        | 단순 번호(ID)      | 희소 벡터(sparse)      | 밀집 벡터(dense)              |
| 차원 크기        | 1                  | 단어 개수만큼          | 보통 수십~수백 차원           |
| 단어 간 유사성   | 반영 불가          | 반영 불가              | 벡터 거리/각도로 반영 가능    |
| 공간 효율        | 높음               | 낮음 (차원 ↑)          | 비교적 높음 (차원 ↓)          |
| 학습 방식        | 규칙만 정의        | 규칙만 정의            | **데이터로부터 학습**         |

- **인코딩**: “텍스트를 숫자로 바꾸는 최소한의 단계”  
- **임베딩**: “숫자 벡터 안에 **의미와 관계**까지 담아내는 표현”

이 차시에서는 텍스트 표현 방법을 발전 흐름에 따라 정리합니다.

- **통계 기반 벡터화(BoW / TF-IDF / N-gram)**  
  단어 빈도와 중요도를 이용해 문서를 숫자 벡터로 표현합니다.

- **분산 표현(Word2Vec)**  
  비슷한 문맥에서 쓰이는 단어가 비슷한 벡터를 갖도록 학습합니다.

- **텍스트 분류 베이스라인**  
  벡터화한 문장을 머신러닝 모델에 넣어 감성 분류 흐름을 완성합니다.


### 1.2 통계 기반 벡터화: BoW / TF-IDF / N-gram 

앞에서 본 것처럼, 우리가 진짜로 원하는 것은  
> 의미가 비슷한 단어는 벡터 공간에서도 가깝게,  
> 다른 의미의 단어는 멀리 떨어지게 만드는 **임베딩(embedding)** 입니다.

그런데 임베딩이 등장하기 전/초기에는,  
텍스트를 숫자로 표현하기 위해 **통계(statistics)에 기반한 벡터화 기법**들이 널리 사용되었습니다.

- 이 기법들은 **“임베딩”이라고 부르기보다는**,  
  “단어/문서의 등장 패턴을 숫자로 표현한 벡터화 방법”에 가깝습니다.
- 즉, **의미를 잘 담은 밀집 임베딩이라기보다는**,  
  “등장 횟수·빈도·조합”을 수치화한 **전통적인 표현 방식**이라고 보는 것이 더 정확합니다.

이 섹션에서는 대표적인 통계 기반 벡터화 기법인 **Bag of Words(BoW), TF-IDF, N-gram**을 간단히 살펴보고,  
이후 섹션에서 등장할 **신경망 기반 임베딩**과 대비해 보겠습니다.

<img src="image/bow_tfidf_ngram_flow.svg" width="760">

이미지 출처: 김민수 강사


#### 1) Bag of Words (BoW)

**아이디어**  
- 문서 안에 어떤 단어들이 **몇 번** 등장했는지만 세어서 벡터로 만드는 방식
- 단어의 **순서나 문맥은 완전히 무시**하고, “가방 속에 단어 개수만 넣어둔다”고 생각하면 됩니다.


```text
문장1: "고양이가 물을 마신다"
문장2: "물이 고양이를 마신다"

어휘 사전: ["고양이", "물", "마신다"]

→ 문장1 BoW: [1, 1, 1]
→ 문장2 BoW: [1, 1, 1]

두 문장의 단어 순서와 의미는 분명히 다르지만,
BoW 벡터는 완전히 동일하게 나옵니다.
```

- 장점:  
  - 구현이 단순하고 직관적
  - 전통적인 문서 분류, 스팸 필터링 등에서 오랫동안 잘 동작해 옴
- 한계:  
  - 단어 **순서, 문맥, 구문 구조**를 전혀 반영하지 못함  
  - 단어 사전 크기만큼 차원이 필요 → **고차원 희소 벡터(sparse vector)**

#### 2) TF-IDF (Term Frequency – Inverse Document Frequency)
TF-IDF는 정보 검색과 텍스트 마이닝에서 널리 쓰이는 단어의 상대적 중요도를 나타내는 통계적 가중치입니다.  
여러 문서(corpus) 중 특정 단어가 개별 문서에서 얼마나 중요한지를 나타내며,  
단순 빈도뿐 아니라 “문서 내 상대적 빈도”와 “전체 문서에서의 희소성”을 함께 고려합니다.

- BoW는 단순히 "단어가 몇 번 나왔는가"만 보지만,  
- TF-IDF는 "문서 내에서의 상대적 중요도 + 전체 문서에서의 희소성"을 함께 고려합니다.  

```
문서 집합에서:
"the"  → 거의 모든 문서에 자주 등장 → 중요도 낮음
"quantum" → 일부 문서에서만 드물게 등장 → 중요도 높음
```

#### 구성 요소

1. **TF (Term Frequency)**  
   - 특정 단어 *t*가 문서 *d* 안에서 얼마나 자주 등장했는지를 나타냄  
   - 단순히 "등장 횟수"를 쓰기도 하고, 문서 길이를 고려해 "등장 비율"로 정규화하기도 함  
   - 자주 나온 단어가 높은 점수  

   **공식**  
   $$
   TF(t, d) = \frac{\text{단어 t가 문서 d에 등장한 횟수}}{\text{문서 d의 전체 단어 수}}
   $$


2. **IDF (Inverse Document Frequency)**  
   - 특정 단어가 전체 문서 집합에서 얼마나 희귀한지를 나타냄  
   - 흔하게 나오는 단어는 IDF가 0에 가까워지고, 드물게 나오는 단어는 IDF가 커짐

   **공식**  
   $$
   IDF(t) = \log \frac{N(전체문서수)}{df(t)}
   $$

   - $ N $: 전체 문서 수  
   - $ df(t) $: 단어 t가 등장한 문서의 수(document frequency)  
   - 즉, 다른 문서에 거의 등장하지 않는 단어일수록 높은 점수



3. **TF-IDF**  
   - 최종적으로 TF와 IDF를 곱해 단어의 중요도를 산출  
   - 다른 문서에 자주 등장하지 않는 단어가 문서A에서 자주 등장했다면, 해당 단어는 문서 A에서 높은 TF-IDF 스코어 획득 

   **공식**  
   $$
   TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)
   $$

#### 계산 예시

문서 집합:
- 문서1: `"나는 고양이를 좋아한다"` (총 3단어)
- 문서2: `"나는 개를 좋아한다"` (총 3단어)  
전체 문서 수 $N=2$  
어휘: {"고양이", "개", "좋아한다", "나는"}

#### Step 1. IDF 계산 (단어의 공통 중요도)

| 단어 | 문서 빈도 (df, Document Frequency) | IDF 계산 (\log(N/df)) | 결과 (\log_{10} 사용) |
| :--- | :--- | :--- | :--- |
| **"고양이"** | 문서1에만 등장 $\rightarrow$ df=1 | $\log (2/1)$ | $\approx 0.301$ |
| **"개"** | 문서2에만 등장 $\rightarrow$ df=1 | $\log (2/1)$ | $\approx 0.301$ |
| **"좋아한다"** | 문서1, 문서2 모두 등장 $\rightarrow$ df=2 | $\log (2/2) = \log 1$ | $0$ |
| **"나는"** | 문서1, 문서2 모두 등장 $\rightarrow$ df=2 | $\log (2/2) = \log 1$ | $0$ |

#### Step 2. 문서1의 TF-IDF 벡터 계산 (문서1: "나는 고양이를 좋아한다")

| 단어 | TF (Term Frequency, 1/3) | IDF (Step 1 결과) | TF-IDF (TF $\times$ IDF) |
| :--- | :--- | :--- | :--- |
| **"고양이"** | $1/3 \approx 0.33$ | $0.301$ | $\approx 0.0993$ |
| **"개"** | $0$ | $0.301$ | $0$ |
| **"좋아한다"** | $1/3 \approx 0.33$ | $0$ | $0$ |
| **"나는"** | $1/3 \approx 0.33$ | $0$ | $0$ |
<br>

**결과: 문서1의 TF-IDF 벡터** $\approx [0.0993, 0, 0, 0]$ (어휘 순)

#### Step 3. 문서2의 TF-IDF 벡터 계산 (문서2: "나는 개를 좋아한다")

| 단어 | TF (Term Frequency, 1/3) | IDF (Step 1 결과) | TF-IDF (TF $\times$ IDF) |
| :--- | :--- | :--- | :--- |
| **"고양이"** | $0$ | $0.301$ | $0$ |
| **"개"** | $1/3 \approx 0.33$ | $0.301$ | $\approx 0.0993$ |
| **"좋아한다"** | $1/3 \approx 0.33$ | $0$ | $0$ |
| **"나는"** | $1/3 \approx 0.33$ | $0$ | $0$ |

**결과: 문서2의 TF-IDF 벡터** $\approx [0, 0.0993, 0, 0]$ (어휘 순)

**최종 결론:**
TF-IDF는 이 과정을 통해 다음을 달성합니다.
1.  **공통 단어("나는", "좋아한다")**: 두 문서에 흔하게 나타나므로 $\text{IDF}$가 $0$이 되어 **중요도 0**을 부여합니다.
2.  **핵심 단어("고양이", "개")**: 해당 문서에서만 특별하게 나타나므로 $\text{IDF}$가 높아져 **높은 중요도**를 부여합니다.

결과적으로, TF-IDF는  
문서1의 특징을 **고양이**로,  
문서2의 특징을 **개**로  
**명확하게 벡터 안에 표현**해 줍니다.

TF-IDF는 통계 기반 벡터화 방식 중 가장 널리 쓰였으며, 특히 다음과 같은 분야에서 핵심적인 역할을 수행했습니다.

1. **정보 검색 (Information Retrieval):**
- 사용자가 검색한 **쿼리(Query)의 중요도**를 계산하고, 웹 문서나 데이터베이스의 **문서 중요도**를 계산하는 데 사용되었습니다. 
- TF-IDF 값이 높은 문서일수록 사용자의 검색 의도와 가장 관련이 높다고 판단하는 방식입니다.
2. **문서 분류 및 클러스터링 (Clustering):**
- 뉴스 기사나 메일 같은 문서들을 TF-IDF 벡터로 변환한 후, 이 벡터들 간의 유사도(거리)를 계산하여   
비슷한 주제의 문서끼리 묶거나(클러스터링),  문서를 특정 주제로 분류하는(Classification)  
**특징 벡터(Feature Vector)** 로 사용되었습니다.

TF-IDF는 단순하지만 강력한 성능을 보여주며, 이후 **단어 중요도 측정** 방식의 기본 토대가 되었습니다.

#### 요약
- TF: 단어가 문서 내에서 얼마나 자주 등장했는가  
- IDF: 단어가 전체 문서 집합에서 얼마나 희귀한가  
- TF-IDF: "문서 내 중요도 × 전체 문서 희소성"  
- 장점: 문서 핵심 키워드 뽑기에 유용, 검색 엔진·문서 추천· 분류에 널리 활용  
- 한계: 여전히 단어 순서·문맥은 반영하지 못함  

#### TF-IDF 실습

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

corpus = [ "I love natural language processing",
          "I love programming in Python",
        "natural language processing is exciting",
        "Harry Potter is a series of seven fantasy fantasy novels"]

tfidf = TfidfVectorizer(stop_words='english').fit(corpus)  # TF-IDF 벡터화 객체입니다.

print(tfidf.vocabulary_)  # 문자열을 정수로 바꿉니다.
pd.DataFrame(tfidf.transform(corpus).toarray(), columns=tfidf.get_feature_names_out())  # 학습된 기준으로 데이터를 변환합니다.

#### 3) N-gram

BoW/TF-IDF는 단어의 **순서**를 완전히 무시한다는 큰 한계가 있습니다.  
이를 조금이라도 보완하기 위해 등장한 것이 **N-gram**입니다.

- 연속된 N개의 단어(또는 문자)를 하나의 묶음으로 보는 방법
- 예: 2-gram(바이그램), 3-gram(트라이그램)

```text
문장: "자연어 처리를 열심히 공부한다"

2-gram (바이그램):
["자연어 처리를", "처리를 열심히", "열심히 공부한다"]
```

- 장점:
  - 연속된 단어 패턴을 반영 → **문맥의 일부**를 표현 가능
  - 예: “뉴욕”, “인공 지능”, “자연어 처리” 같은 구를 잘 잡아낼 수 있음
- 한계:
  - N을 크게 할수록 경우의 수가 폭발 → 차원 수가 매우 커짐
  - 희귀하게 등장하는 N-gram은 데이터가 거의 없어 다루기 어려움
  - 여전히 **의미/유사도**를 직접적으로 학습한다기보다는, “공동 등장 패턴”에 가깝다

#### 4) 통계 기반 벡터화 정리

지금까지 본 BoW / TF-IDF / N-gram은

- 텍스트를 벡터로 만드는 **전통적인 통계 기반 표현 방식**이고,
- “벡터화”는 했지만, 우리가 3.1에서 정의한  
  **“의미를 잘 반영하는 밀집 임베딩”과는 거리가 있습니다.**

정리하자면,

- 장점:
  - 단순하고 구현이 쉽다
  - 문서 분류, 키워드 추출, 검색 등 고전적인 NLP 태스크에서 여전히 유용하다
- 근본적인 한계:
  - 단어 순서, 문맥, 깊은 의미를 잘 반영하지 못한다
  - 고차원 희소 벡터가 되어 계산·메모리 효율이 떨어진다
  - 비슷한 의미의 단어를 **벡터 공간에서 가깝게 모으는 구조가 아니다**

이런 이유로,  
> “단어/문장의 **의미**를 벡터 공간에 잘 담아내고 싶다”는 요구가 커지면서  
> **신경망 기반의 임베딩(Embedding Layer, Word2Vec 등)** 이 본격적으로 사용되기 시작했습니다.

이제 단어의 의미를 벡터 공간에 학습하는 대표적인 방법인  
**Word2Vec**의 기본 아이디어를 살펴보겠습니다.


#### 문제 1. TF-IDF
문장 리스트 `["나는 NLP를 좋아한다", "NLP는 재미있다"]`를 TF-IDF 벡터로 변환하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.

docs = ["나는 NLP를 좋아한다", "NLP는 재미있다"]  # 벡터화할 문장 목록입니다.
vectorizer = TfidfVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print(vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X.toarray())  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.

docs = ["나는 NLP를 좋아한다", "NLP는 재미있다"]

#### 문제 2. BoW 벡터 만들기
문장 리스트 `["나는 밥을 먹었다", "나는 물을 마셨다"]`에서 **Bag of Words** 벡터를 만들어보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import CountVectorizer  # 문장을 단어 개수 벡터로 바꾸는 도구입니다.

docs = ["나는 밥을 먹었다", "나는 물을 마셨다"]  # 벡터화할 문장 목록입니다.
vectorizer = CountVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print("단어 사전:", vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print("BoW 벡터:\n", X.toarray())  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터화 도구입니다.

docs = ["나는 밥을 먹었다", "나는 물을 마셨다"]

#### 문제 3. TF-IDF 계산
문장 리스트 `["나는 NLP를 공부한다", "NLP는 재미있다", "공부는 힘들다"]`를 **TF-IDF**로 벡터화하세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.

docs = ["나는 NLP를 공부한다", "NLP는 재미있다", "공부는 힘들다"]  # 벡터화할 문장 목록입니다.
vectorizer = TfidfVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print("단어 사전:", vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print("TF-IDF 벡터:\n", X.toarray())  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.

docs = ["나는 NLP를 공부한다", "NLP는 재미있다", "공부는 힘들다"]

#### 문제 4. N-gram 적용
문장 `"자연어 처리를 공부한다"`에 대해 **2-gram(바이그램)** 을 추출해보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import CountVectorizer  # 문장을 단어 개수 벡터로 바꾸는 도구입니다.

docs = ["자연어 처리를 공부한다"]  # 벡터화할 문장 목록입니다.
vectorizer = CountVectorizer(ngram_range=(2,2))  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print("바이그램 사전:", vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print("바이그램 벡터:", X.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
# 여기에 작성하세요
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터화 도구입니다.

docs = ["자연어 처리를 공부한다"]

## 2. 분산 표현: Word2Vec

임베딩 레이어는 **딥러닝 모델 안에 포함된 임베딩 테이블**이었습니다.  
이번에는 임베딩만 따로 학습하는 대표적인 신경망 기법인 **Word2Vec**을 살펴봅니다.


#### 1) 분산 표현(Distributed Representation)이란?

기존 통계 기반 벡터화(BoW, TF-IDF)는

- 단어마다 “등장 횟수/빈도/조합”을 세어 벡터로 만들었습니다.
- 단어 의미나 문맥을 직접 학습한다기보다, **통계를 정리한 결과**에 가깝습니다.

반면, 분산 표현은

> “비슷한 문맥에서 쓰이는 단어는 비슷한 벡터를 갖도록”  
> 신경망으로 **벡터 자체를 학습**하는 접근입니다.  
> (의미가 여러 차원에 분산되어 있다는 뜻. 원핫은 오직 한 차원('1'이 있는 그 위치)에만 정보가 있음)

- 각 단어는 보통 수십~수백 차원의 **밀집 벡터(dense vector)** 로 표현되고,
- 벡터 공간에서의 위치/거리/각도가 의미 정보를 반영하도록 학습됩니다.

#### 2) Word2Vec

Word2Vec은 **아주 얕은 신경망**을 사용해 단어 임베딩을 학습하는 모델입니다.

핵심 아이디어:

- 단어는 **주변 단어(문맥)** 과 함께 나타납니다.
- “비슷한 문맥에서 등장하는 단어들끼리는 의미가 비슷할 가능성이 크다.”
- 따라서, 문맥 예시들을 많이 보여주면서  
  임베딩이 그 패턴을 잘 예측하도록 학습합니다.

주요 구조는 두 가지입니다. 먼저 큰 방향을 보면, CBOW와 Skip-Gram은 **예측 방향**이 서로 반대입니다.

<img src="image/word2vec_cbow_skipgram_flow.svg" width="760">

이미지 출처: 김민수 강사

아래 그림은 같은 내용을 신경망 구조에 조금 더 가깝게 보여줍니다.

<table>
  <tr>
    <td align="center"><img src="image/word2vec_cbow_commons.svg" width="360"></td>
    <td align="center"><img src="image/word2vec_skipgram_commons.svg" width="360"></td>
  </tr>
  <tr>
    <td align="center"><b>CBOW</b>: 주변 단어로 중심 단어 예측</td>
    <td align="center"><b>Skip-Gram</b>: 중심 단어로 주변 단어 예측</td>
  </tr>
</table>

외부 이미지 출처: Wikimedia Commons, [Continuous Bag of Words model (CBOW).svg](https://commons.wikimedia.org/wiki/File:Continuous_Bag_of_Words_model_(CBOW).svg), [Skip-gram.svg](https://commons.wikimedia.org/wiki/File:Skip-gram.svg) / 저자: Zhang, Aston; Lipton, Zachary C.; Li, Mu; Smola, Alexander J. / 라이선스: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) / 원본 SVG를 파일명만 바꿔 사용

- **CBOW (Continuous Bag of Words)**  
  - 주변 단어들(컨텍스트) → 중심 단어를 예측  
  - 예: `"나는 ___을 먹었다"` → 빈칸에 들어갈 단어를 맞히도록 학습
- **Skip-Gram**  
  - 중심 단어 → 주변 단어들을 예측  
  - 예: `"밥"` → (나는, 먹었다) 같은 주변 단어들을 맞히도록 학습

학습이 잘 되면, 임베딩 벡터 공간에서  
재미있는 **벡터 연산**이 가능해지는 것으로 유명합니다.

```text
벡터("왕") - 벡터("남자") + 벡터("여자") ≈ 벡터("여왕")
```

<img src="image/word_vector_illustration_commons.jpg" width="460">

외부 이미지 출처: Wikimedia Commons, [Word vector illustration.jpg](https://commons.wikimedia.org/wiki/File:Word_vector_illustration.jpg) / 저자: Singerep / 라이선스: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) / 원본 이미지를 파일명만 바꿔 사용

즉, 단어들 사이의 관계(성별, 직업, 국가 등)가  
벡터 공간에서 **산술 연산**으로도 어느 정도 드러납니다. 


#### 3) Word2Vec vs 임베딩 레이어

Word2Vec과 임베딩 레이어는 모두 단어를 의미 벡터로 표현하지만, 학습되는 위치가 다릅니다.

| 구분 | Word2Vec | 임베딩 레이어 |
|------|----------|---------------|
| 학습 위치 | 임베딩만 따로 사전 학습 | 분류·번역 같은 모델 안에서 함께 학습 |
| 학습 신호 | 주변 단어 예측 | 최종 태스크의 정답 |
| 활용 방식 | 학습된 단어 벡터를 다른 모델에 재사용 | 특정 모델에 맞게 계속 조정 |

실제 프로젝트에서는 미리 학습된 Word2Vec 벡터를 임베딩 레이어의 초기값으로 불러와 사용하는 방식도 있습니다.


## 3. 현대 NLP에서의 임베딩: "고정된 벡터"에서 "계산되는 표현"으로

우리가 지금까지 배운 Word2Vec은 단어마다 하나의 고유한 벡터 좌표를 할당하는 **정적(Static) 임베딩**이었습니다.  
하지만 현대 NLP는 여기서 한 단계 더 진보하여, 단어의 벡터를 문장 속에서 실시간으로 만들어내는 **문맥적 임베딩(Contextual Embedding)** 시대로 넘어왔습니다.

<img src="image/static_vs_contextual_embedding.svg" width="760">

이미지 출처: 김민수 강사

#### 1) 작동 방식의 변화: "찾아보기"에서 "실시간 합성"으로

가장 큰 차이는 임베딩이 결정되는 **시점**과 **방식**에 있습니다.

- **기존 방식 (Word2Vec):** - 메모리에 저장된 **임베딩 표(Lookup Table)** 에서 단어에 맞는 벡터를 꺼내오면 끝입니다.
- 예를 들어 "은행"은 어떤 문장에서도 항상 `[0.1, -0.5]` 같은 고정된 값입니다.
- **현대 방식 (Transformer/BERT 등):** - 임베딩 레이어에서 단어의 기본 벡터를 가져온 뒤,  
**신경망 층(Encoder)** 을 통과하며 주변 단어들의 정보를 수치적으로 **합산(Combine)** 합니다.
- 예를 들어, `"사과를 먹다"`라는 문장에서 '사과' 벡터는 옆에 있는 '먹다'라는 단어의 정보를 흡수하여  
**음식**의 특징이 실시간으로 강화된 새로운 벡터로 변신합니다.


#### 2) 학습 메커니즘의 진화: 빈칸 채우기(Masking)를 통한 문맥 파악

현대 모델은 단순히 단어의 짝을 맞히는 것을 넘어, 문장 전체의 구조를 이해하기 위해 훨씬 고차원적인 방식으로 사전 학습(Pre-training)됩니다.

- **마스크 학습 (Masked Language Modeling):**
  - 문장 중간의 단어를 가려놓고(`[MASK]`), 주변 단어들을 조합해 그 빈칸에 들어갈 단어를 맞히는 연습을 합니다.
  - 예: `"나는 오늘 [MASK]에 가서 돈을 입금했다"` → 주변의 '돈', '입금'을 보고 `[MASK]`가 '은행'임을 추론해야 합니다.
- **문맥 정보의 내재화:** 
  - 이 빈칸을 맞히기 위해 모델은 **"주변에 어떤 단어가 올 때 이 단어의 의미가 어떻게 변하는지"** 그 관계(Attention)를 아주 정교하게 학습하게 됩니다.
  - 이 과정이 수억 개의 문장에 대해 반복되면서, 단어 벡터는 단순한 의미를 넘어 **문맥을 읽어내는 능력**을 갖추게 됩니다.

#### 3) 정적 임베딩 vs 문맥적 임베딩 비교 요약

| 구분 | 정적 임베딩 (Word2Vec) | 문맥적 임베딩 (BERT, GPT 등) |
| :--- | :--- | :--- |
| **핵심 원리** | 미리 저장된 벡터를 **꺼내오기(Lookup)** | 주변 단어와 정보를 **섞어서 계산(Compute)** |
| **다의어 처리** | "은행"의 모든 의미가 하나의 벡터에 섞임 | 문맥(주변 단어)을 보고 실시간으로 의미 분리 |
| **학습 방식** | 단어 간의 단순 통계적 유사성 학습 | **빈칸 채우기(Masking)** 등으로 문맥 간 관계 학습 |
| **구성 요소** | 단어의 고유 의미 | **토큰 의미 + 순서(Position) + 문맥 정보** |

**정리:** 현대 NLP 임베딩의 핵심은 **학습 시점에 빈칸 채우기 등을 통해 단어들 사이의 고차원적인 관계를 미리 파악하고,  
이를 바탕으로 실제 문장에서 단어의 의미를 실시간으로 계산해낸다**는 점에 있습니다.

이러한 '실시간 계산 엔진'의 정체인 **Transformer**와 **Attention**의 구조를 다음 장에서 본격적으로 파헤쳐 보겠습니다.


#### 정리
- **원-핫 인코딩**: 단어 구분만 가능, 유사성 반영 불가  
- **통계 기반 임베딩**: 빈도·중요도·순서를 반영하지만, 의미는 여전히 부족  
- **분산 표현(Word2Vec)**: 신경망을 통해 **단어 간 의미 관계**까지 수치화  
- → 이는 이후 문맥적 임베딩과 LLM을 이해하는 기초 표현으로 연결됨 


#### 문제 5. Word2Vec 학습
문장 리스트 `["나는 학교에 간다", "학교에서 공부한다", "나는 밥을 먹었다"]`로 Word2Vec 모델을 학습한 뒤, `"학교"`와 가장 유사한 단어를 찾아보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from gensim.models import Word2Vec  # Word2Vec 임베딩 모델입니다.

sentences = [  # 임베딩 학습에 사용할 문장 목록입니다.
    ["나는", "학교에", "간다"],
    ["학교에서", "공부한다"],
    ["나는", "밥을", "먹었다"],
    ["좋은", "학교", "수업"],
    ["맛있는", "밥"],
    ["시원한", "콜라를","마셨다"],
]

model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=0)  # Word2Vec 임베딩 모델입니다. vector_size는 벡터 크기입니다.

print("학교와 가장 유사한 단어:", model.wv.most_similar("학교", topn=1))  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요
from gensim.models import Word2Vec  # Word2Vec 임베딩 모델입니다.

sentences = [
    ["나는", "학교에", "간다"],
    ["학교에서", "공부한다"],
    ["나는", "밥을", "먹었다"],
    ["좋은", "학교", "수업"],
    ["맛있는", "밥"],
    ["시원한", "콜라를","마셨다"],
]

#### 문제 6. Word2Vec 벡터 연산
Word2Vec을 학습한 모델에서 `"밥"` - `"먹었다"` + `"마셨다"` ≈ ? 를 계산해보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

print(model.wv.most_similar(positive=["밥", "마셨다"], negative=["먹었다"], topn=1))  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
# 여기에 작성하세요


#### 문제 7. Word2Vec 학습 방식 바꾸기

같은 문장 리스트로 `sg=1`을 지정해 Skip-Gram 방식의 Word2Vec 모델을 학습해 보세요. 학습한 뒤 `"밥"`과 가장 유사한 단어를 확인하세요.

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

skipgram_model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=1)  # sg=1은 Skip-Gram 방식입니다.

print("밥과 가장 유사한 단어:", skipgram_model.wv.most_similar("밥", topn=1))  # 결과를 화면에 출력합니다.
```
</details>


In [ ]:
# 여기에 작성하세요


## 4. 텍스트 분류 미니 프로젝트

짧은 리뷰 문장을 벡터화하고, 고전적인 머신러닝 모델로 감성 분류 베이스라인을 만들어 봅니다.

- 목표: `전처리 -> 벡터화 -> 학습 -> 해석` 흐름을 한 번에 연결합니다.
- 사용 도구: `CountVectorizer`, `TfidfVectorizer`, `LogisticRegression`

<img src="image/text_classification_pipeline.svg" width="760">

이미지 출처: 김민수 강사


In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

reviews = [
    ("배송이 빨라서 만족합니다", 1),
    ("포장이 깔끔하고 품질도 좋아요", 1),
    ("생각보다 성능이 좋아서 추천합니다", 1),
    ("가격 대비 괜찮은 선택이었어요", 1),
    ("다시 사고 싶지 않을 만큼 아쉬워요", 0),
    ("설명과 달라서 많이 실망했습니다", 0),
    ("배송이 늦고 응대도 아쉬웠어요", 0),
    ("품질이 기대보다 떨어집니다", 0),
]

df = pd.DataFrame(reviews, columns=["text", "label"])  # 데이터프레임을 직접 만듭니다.
df


### 문제 8. CountVectorizer로 문장 행렬 만들기

위의 `df["text"]`를 `CountVectorizer`로 변환하고, 단어 목록과 문서-단어 행렬을 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import CountVectorizer  # 문장을 단어 개수 벡터로 바꾸는 도구입니다.

count_vectorizer = CountVectorizer()  # 단어 빈도 벡터화 객체입니다.
X_count = count_vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.

print(count_vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X_count.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터화 도구입니다.

# 여기에 작성하세요


### 문제 9. TfidfVectorizer로 다시 표현하기

같은 데이터를 `TfidfVectorizer`로 바꾸고, `CountVectorizer`와 어떤 차이가 보이는지 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.

tfidf_vectorizer = TfidfVectorizer()  # TF-IDF 벡터화 객체입니다.
X_tfidf = tfidf_vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.

print(tfidf_vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X_tfidf.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.

# 여기에 작성하세요


### 문제 10. 로지스틱 회귀로 감성 분류 베이스라인 만들기

`TfidfVectorizer`와 `LogisticRegression`을 이용해 간단한 감성 분류 베이스라인을 만들어 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 모델입니다.
from sklearn.metrics import accuracy_score  # 정확도를 계산하는 함수입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터를 나누는 함수입니다.
from sklearn.pipeline import make_pipeline  # 파이프라인 생성 함수입니다.

X_train, X_test, y_train, y_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    df["text"], df["label"], test_size=0.25, random_state=42  # 새 파생 변수를 만들어 저장합니다.
)

model = make_pipeline(  # 전처리와 모델 파이프라인을 만듭니다.
    TfidfVectorizer(),  # TF-IDF 벡터화 객체입니다.
    LogisticRegression(max_iter=1000, random_state=42),  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
)

model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
pred = model.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("accuracy:", accuracy_score(y_test, pred))  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.metrics import accuracy_score  # 정확도 지표 함수입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.pipeline import make_pipeline  # 파이프라인 생성 함수입니다.

# 여기에 작성하세요


### 문제 11. 분류 결과에 영향을 많이 준 단어 확인하기

학습된 로지스틱 회귀 모델에서 긍정/부정 예측에 큰 영향을 준 단어를 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

vectorizer = TfidfVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.
y = df["label"]  # 정답 값 또는 두 번째 배열을 준비합니다.

clf = LogisticRegression(max_iter=1000, random_state=42)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
clf.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

feature_names = vectorizer.get_feature_names_out()  # 벡터화된 단어 목록을 가져옵니다.
coef = clf.coef_[0]  # 각 단어의 분류 가중치를 가져옵니다.

top_positive = coef.argsort()[-5:][::-1]  # 긍정 방향 가중치가 큰 단어 인덱스입니다.
top_negative = coef.argsort()[:5]  # 부정 방향 가중치가 큰 단어 인덱스입니다.

print("긍정에 가까운 단어:", feature_names[top_positive])  # 결과를 화면에 출력합니다.
print("부정에 가까운 단어:", feature_names[top_negative])  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
# 여기에 작성하세요


## 체크포인트

- 텍스트 전처리 방법은 데이터 목적에 따라 달라집니다.
- BoW, TF-IDF, Word2Vec은 모두 텍스트를 수치화하는 방법이지만 표현 방식과 장단점이 다릅니다.
- 딥러닝 모델을 쓰지 않아도 `벡터화 + 선형 모델`만으로 텍스트 분류의 기본 흐름을 만들 수 있습니다.
